# 11. RQ5: Cross-Domain Effect-Size Synthesis

**RQ5:** Is the magnitude of governance-relevant change identified in the software
QA domain (RQ1/RQ2 era interaction effects) comparable to the magnitude of change
identified in the IT audit domain (RQ4's driver-set shift relative to Mojtahedi &
Zhou, 2024) -- suggesting a shared "digital-era governance disruption" -- or do the
two domains diverge?

**Method:** Side-by-side comparison of standardized effect sizes (Cohen's f-squared)
across both domains.

---
## Status: NOT YET RUNNABLE -- scaffold only, not executed

This notebook depends on a real output from `10_rq4_remediation_time_model.ipynb`
(RQ4's effect size), which cannot currently run (see that notebook's status note --
it needs real SEC EDGAR data that has not been extracted yet). RQ5 is therefore the
last research question in the whole project to become runnable, since it is
downstream of RQ4.

**The QA-domain side of this comparison is now fully real**, from two notebooks:
- `07_rq1_code_metric_mining_analysis.ipynb` -- real RQ1 effect size from mined
  Apache Camel/Hadoop code metrics (McFadden pseudo-R2 improves from 0.003 to
  0.018 with era interactions)
- `08_rq2_era_analysis.ipynb` -- real RQ2 effect size, Cohen's f2 = 0.0011
  (negligible) for the resolution-time era interaction

There is still no equivalent real number for the audit-domain side, since that
requires RQ4's SEC EDGAR-based remediation model. RQ5's actual cross-domain
comparison cannot be completed until that exists.

The original code is preserved below, unmodified, for reference and so it's ready
to run the moment RQ4 has a real, completed model.


In [1]:
!pip install -q pandas statsmodels || pip install -q pandas statsmodels --break-system-packages

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## Original scaffold code (will raise a file-not-found error if run today)

In [2]:
import pandas as pd
import statsmodels.formula.api as smf


def cohens_f_squared(r2_full: float, r2_reduced: float) -> float:
    """
    Cohen\'s f^2 = (R2_full - R2_reduced) / (1 - R2_full)
    Conventions (Cohen, 1988): 0.02 = small, 0.15 = medium, 0.35 = large
    """
    return (r2_full - r2_reduced) / (1 - r2_full)


def interpret_effect_size(f2: float) -> str:
    if f2 < 0.02:
        return "negligible"
    elif f2 < 0.15:
        return "small"
    elif f2 < 0.35:
        return "medium"
    else:
        return "large"


def get_qa_domain_effect_size(df: pd.DataFrame) -> float:
    """Compare R2 of the QA model WITH era interactions vs. WITHOUT them."""
    df = df.dropna(subset=["resolution_time_days", "num_comments",
                            "num_reassignments", "priority", "era"])
    df["era_binary"] = (df["era"] == "ai_era").astype(int)

    full_model = smf.ols(
        "resolution_time_days ~ (num_comments + num_reassignments + "
        "C(priority)) * era_binary", data=df
    ).fit()
    reduced_model = smf.ols(
        "resolution_time_days ~ num_comments + num_reassignments + C(priority) + era_binary",
        data=df
    ).fit()

    f2 = cohens_f_squared(full_model.rsquared, reduced_model.rsquared)
    print(f"QA domain (RQ1/RQ2): R2_full={full_model.rsquared:.4f}, "
          f"R2_reduced={reduced_model.rsquared:.4f}, f2={f2:.4f} "
          f"({interpret_effect_size(f2)} effect)")
    return f2


def get_audit_domain_effect_size(df: pd.DataFrame) -> float:
    """
    Compare R2 of the 2020+ remediation-time model WITH the full driver set
    vs. a model restricted to only the drivers reported as significant in
    Mojtahedi & Zhou (2024).
    """
    df = df.dropna(subset=["remediation_days", "weakness_category",
                            "industry_sic", "inspection_year"])

    full_model = smf.ols(
        "remediation_days ~ C(weakness_category) + C(industry_sic) + inspection_year",
        data=df
    ).fit()

    prior_literature_model = smf.ols(
        "remediation_days ~ C(weakness_category)", data=df
    ).fit()

    f2 = cohens_f_squared(full_model.rsquared, prior_literature_model.rsquared)
    print(f"Audit domain (RQ4): R2_full={full_model.rsquared:.4f}, "
          f"R2_prior_literature_only={prior_literature_model.rsquared:.4f}, "
          f"f2={f2:.4f} ({interpret_effect_size(f2)} effect)")
    return f2

In [3]:
# NOT EXECUTED -- ../data/cleaned/qa_defect_dataset.csv is missing num_reassignments,
# and ../data/cleaned/audit_disclosure_dataset.csv does not exist yet with
# remediation_days / weakness_category / industry_sic populated.
#
# qa_df = pd.read_csv("../data/cleaned/qa_defect_dataset.csv")
# audit_df = pd.read_csv("../data/cleaned/audit_disclosure_dataset.csv")
#
# qa_f2 = get_qa_domain_effect_size(qa_df)
# audit_f2 = get_audit_domain_effect_size(audit_df)
#
# print("\n=== RQ5: Cross-Domain Comparison ===")
# print(f"QA domain effect size (f2):    {qa_f2:.4f} ({interpret_effect_size(qa_f2)})")
# print(f"Audit domain effect size (f2): {audit_f2:.4f} ({interpret_effect_size(audit_f2)})")
#
# diff = abs(qa_f2 - audit_f2)
# print(f"\nAbsolute difference in effect size: {diff:.4f}")
# if diff < 0.10:
#     print("Interpretation: comparable magnitudes -> supports a shared "
#           "\'digital-era governance disruption\' across both domains (H5 supported).")
# else:
#     print("Interpretation: divergent magnitudes -> governance-relevant "
#           "disruption appears domain-specific rather than shared (H5 not supported).")

print("Scaffold only -- see markdown cell above for what is blocking this notebook.")
print("Known so far: QA-domain (RQ2) real effect size = 0.0011 (negligible), from 08_rq2_era_analysis.ipynb and 07_rq1_code_metric_mining_analysis.ipynb.")
print("Audit-domain (RQ4) effect size: not yet available.")

Scaffold only -- see markdown cell above for what is blocking this notebook.
Known so far: QA-domain (RQ2) real effect size = 0.0011 (negligible), from 08_rq2_era_analysis.ipynb and 07_rq1_code_metric_mining_analysis.ipynb.
Audit-domain (RQ4) effect size: not yet available.
